## Before Starting (If you haven't done it already)
Go to https://aistudio.google.com/app/apikey copy generative language client free tier API key

After you did that click the key icon at the left sidebar add your API key with GOOGLE_API_KEY as name and your API key as the value and enable notebook access

## Setup

### Install dependencies

In [ ]:
%pip install -qU 'google-genai>=1.0.0'
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.1/226.1 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.4 MB/s eta 0:00:00


### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY3")
client = genai.Client(api_key=GOOGLE_API_KEY)

### Choose a model

Different models have different ups and downs.

In [ ]:
MODEL_ID="gemini-2.5-flash" # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", ""] {"allow-input":true, isTemplate: true}

## Robot Assistant Feature Implementation Plan

### Features to Implement

---

#### 1. **Dynamic Promotion Announcer**
- **Description:** Automatically announces real-time promotions (e.g., "Last 3 bagels – 30% off!") triggered by inventory status or timed events.
- **Trigger Sources:**
  - Low stock threshold
  - Scheduled time-sale events
- **Goal:** Drive quick conversion on perishable or surplus items.

---

#### 2. **Personality Modulation**
- **Description:** Adjusts the robot's speaking tone and behavior based on time of day.
- **Modes:**
  - Morning shift: Energetic, friendly
  - Afternoon shift: Standart
  - Evening shift: Calmer, relaxed
- **Goal:** Improve customer comfort and reinforce brand personality throughout the day.

---

#### 3. **User Memory**
- **Description:** Stores past interactions to personalize future conversations and recommendations.
- **Includes:**
  - Purchase history
  - Favorite items
  - Previous conversations or requests
- **Goal:** Create continuity, repeat customer recognition, and context-aware responses.

---

#### 4. **Product Recommendation System**
- **Description:** Recommends 3 products in real time based on:
  - User preferences
  - Current inventory and freshness
  - Sales data or popularity
- **Output:** Products are visually displayed and described, optionally with gestures.
- **Goal:** Boost sales through contextual upselling.

---

#### 5. **Function Calling Layer**
- **Description:** Connects the LLM to the robot’s physical and interface actions.
- **Functionality:**
  - Pointing to specific items
  - Waving or nodding
  - Activating display cards
  - Storytelling mode with gestures
- **Includes:** Support for a **Children’s Interactive Story Mode**
  - Tells bakery-themed stories interactively
  - Uses gestures and voice to engage young customers
- **Goal:** Create an engaging, memorable in-store experience.

---


In [ ]:
# === Bread Inventory, Users, and Robot State Setup ===
import random
from datetime import datetime, timedelta
import pandas as pd
from IPython.display import display

days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

current_day = random.choice(days)
print("Current day:",current_day)
# --- Define 20 breads with metadata ---
breads = {
    "sourdough": {"description": "A tangy, crusty bread made with natural fermentation.", "tags": ["savory", "crusty", "artisan"], "calories": 170, "price": 4.50},
    "baguette": {"description": "A long, thin French bread with a crisp crust.", "tags": ["crusty", "classic", "savory"], "calories": 150, "price": 3.00},
    "challah": {"description": "A sweet, braided Jewish bread often eaten on holidays.", "tags": ["sweet", "soft", "braided"], "calories": 240, "price": 5.00},
    "brioche": {"description": "A rich, buttery bread with a light, puffy texture.", "tags": ["sweet", "buttery", "soft"], "calories": 290, "price": 4.75},
    "ciabatta": {"description": "An Italian white bread with a crisp crust and airy interior.", "tags": ["crusty", "airy", "savory"], "calories": 180, "price": 3.50},
    "rye": {"description": "Dense bread made with rye flour, often slightly sour.", "tags": ["dense", "savory", "traditional"], "calories": 165, "price": 4.25},
    "whole_wheat": {"description": "Nutritious bread made from whole wheat flour.", "tags": ["healthy", "dense", "savory"], "calories": 120, "price": 3.25},
    "multigrain": {"description": "Bread made with a mix of grains and seeds.", "tags": ["healthy", "seeded", "savory"], "calories": 130, "price": 3.75},
    "focaccia": {"description": "Italian flatbread often topped with herbs and olive oil.", "tags": ["flatbread", "herbed", "savory"], "calories": 220, "price": 4.00},
    "naan": {"description": "A soft, leavened Indian flatbread, perfect with curries.", "tags": ["flatbread", "soft", "savory"], "calories": 260, "price": 2.75},
    "pita": {"description": "A Middle Eastern round bread with a pocket.", "tags": ["pocket", "flatbread", "savory"], "calories": 165, "price": 2.50},
    "english_muffin": {"description": "A round, flat bread with a chewy texture.", "tags": ["chewy", "toasted", "savory"], "calories": 130, "price": 2.25},
    "cornbread": {"description": "Moist bread made from cornmeal, slightly sweet.", "tags": ["sweet", "moist", "corn-based"], "calories": 200, "price": 3.00},
    "banana_bread": {"description": "Sweet, moist bread made with ripe bananas.", "tags": ["sweet", "moist", "fruit"], "calories": 260, "price": 3.50},
    "zucchini_bread": {"description": "Sweet, spiced bread made with shredded zucchini.", "tags": ["sweet", "vegetable", "moist"], "calories": 230, "price": 3.50},
    "olive_loaf": {"description": "Savory bread filled with olives and herbs.", "tags": ["savory", "herbed", "olive"], "calories": 190, "price": 4.75},
    "cinnamon_roll": {"description": "Sweet rolled bread filled with cinnamon and sugar.", "tags": ["sweet", "dessert", "swirled"], "calories": 350, "price": 2.95},
    "pretzel_bread": {"description": "Dense, chewy bread with a deep brown crust.", "tags": ["chewy", "dense", "savory"], "calories": 210, "price": 2.85},
    "pumpkin_bread": {"description": "Moist, spiced bread made with pumpkin puree.", "tags": ["sweet", "moist", "seasonal"], "calories": 240, "price": 3.75},
    "milk_bread": {"description": "Ultra-soft and slightly sweet Japanese-style bread.", "tags": ["soft", "sweet", "airy"], "calories": 180, "price": 3.60}
}

# --- Enhance bread metadata with inventory logic ---
for bread in breads:
    breads[bread]["stock"] = random.randint(0, 10)
    breads[bread]["bake_date"] = (datetime.now() - timedelta(days=random.randint(0, 2))).strftime("%Y-%m-%d")
    breads[bread]["shelf_life_days"] = random.choice([1, 2, 3])
    breads[bread]["on_sale"] = False
    breads[bread]["sales_count"] = random.randint(0, 20)

# --- Define 8 users with history and preferences ---
users = {
    "user_001": {"name": "Alice", "preferences": ["sweet", "soft"], "purchase_history": ["brioche", "banana_bread", "challah"], "last_seen": "2025-07-02T09:45:00", "visits": 4, "child_mode": False},
    "user_002": {"name": "Bob", "preferences": ["savory", "crusty"], "purchase_history": ["baguette", "sourdough", "ciabatta"], "last_seen": "2025-07-01T17:20:00", "visits": 7, "child_mode": False},
    "user_003": {"name": "Charlie", "preferences": ["healthy", "seeded"], "purchase_history": ["multigrain", "whole_wheat"], "last_seen": "2025-06-30T13:15:00", "visits": 2, "child_mode": False},
    "user_004": {"name": "Diana", "preferences": ["sweet", "moist"], "purchase_history": ["banana_bread", "zucchini_bread", "pumpkin_bread"], "last_seen": "2025-06-29T10:00:00", "visits": 5, "child_mode": False},
    "user_005": {"name": "Eli", "preferences": ["flatbread", "savory"], "purchase_history": ["naan", "pita", "focaccia"], "last_seen": "2025-07-01T18:30:00", "visits": 3, "child_mode": False},
    "user_006": {"name": "Fatima", "preferences": ["airy", "buttery"], "purchase_history": ["milk_bread", "brioche"], "last_seen": "2025-06-28T16:45:00", "visits": 6, "child_mode": False},
    "user_007": {"name": "George", "preferences": ["chewy", "dense"], "purchase_history": ["pretzel_bread", "rye", "english_muffin"], "last_seen": "2025-07-02T08:50:00", "visits": 8, "child_mode": False},
    "user_008": {"name": "Hannah", "preferences": ["dessert", "sweet", "swirled"], "purchase_history": ["cinnamon_roll", "pumpkin_bread"], "last_seen": "2025-07-01T11:10:00", "visits": 5, "child_mode": False}
}

# Add conversation log for memory
for user in users.values():
    user["conversation_log"] = []



# --- Apply promotion logic for Friday/Sunday ---
def apply_promotions(current_day):
    for bread in breads:
        original_price = breads[bread].get("original_price", breads[bread]["price"])
        breads[bread]["original_price"] = original_price  # persist original price

        if current_day.lower() in ["friday", "sunday"] or  breads[bread]['shelf_life_days']<=1:
            breads[bread]["on_sale"] = True
            breads[bread]["price"] = round(original_price * 0.7, 2)  # 30% off
        else:
            breads[bread]["on_sale"] = False
            breads[bread]["price"] = original_price  # reset to full price


apply_promotions(current_day)  # Set day here
#random timeofday morning, afternoon, night
time_of_day = random.choice(["morning", "night"])
# Convert the breads dictionary to a DataFrame for display
df_breads_full = pd.DataFrame.from_dict(breads, orient="index")
df_breads_full.reset_index(inplace=True)
df_breads_full.rename(columns={'index': 'bread_name'}, inplace=True)

display(df_breads_full)




Current day: saturday


,bread_name,description,tags,calories,price,stock,bake_date,shelf_life_days,on_sale,sales_count,original_price
0,sourdough,"A tangy, crusty bread made with natural fermen...","[savory, crusty, artisan]",170,3.15,5,2025-07-04,1,True,14,4.50
1,baguette,"A long, thin French bread with a crisp crust.","[crusty, classic, savory]",150,3.00,5,2025-07-04,3,False,16,3.00
2,challah,"A sweet, braided Jewish bread often eaten on h...","[sweet, soft, braided]",240,5.00,9,2025-07-02,3,False,4,5.00
3,brioche,"A rich, buttery bread with a light, puffy text...","[sweet, buttery, soft]",290,4.75,10,2025-07-03,2,False,2,4.75
4,ciabatta,An Italian white bread with a crisp crust and ...,"[crusty, airy, savory]",180,3.50,9,2025-07-02,2,False,8,3.50
5,rye,"Dense bread made with rye flour, often slightl...","[dense, savory, traditional]",165,4.25,8,2025-07-04,3,False,17,4.25
6,whole_wheat,Nutritious bread made from whole wheat flour.,"[healthy, dense, savory]",120,2.27,1,2025-07-04,1,True,7,3.25
7,multigrain,Bread made with a mix of grains and seeds.,"[healthy, seeded, savory]",130,2.62,10,2025-07-03,1,True,14,3.75
8,focaccia,Italian flatbread often topped with herbs and ...,"[flatbread, herbed, savory]",220,4.00,1,2025-07-03,2,False,7,4.00
9,naan,"A soft, leavened Indian flatbread, perfect wit...","[flatbread, soft, savory]",260,2.75,7,2025-07-02,3,False,9,2.75


In [ ]:
from datetime import datetime, timedelta
import random
from typing import List, Dict, Any

def point_to_bread(bread: str) -> str:
    """
    Make the robot point to a specific bread on the shelf.
    """
    print(f'Function call:   Robot points to {bread.replace("_", " ").title()}.')
    return f"  Robot points to {bread.replace('_', ' ').title()}."

def reccomend_breads() -> str:
    """
    Description: Recommends 3 products in real time based on:
    User preferences
    Current inventory and freshness
    Sales data or popularity

    Reccomendations should work like the folloiwng:
    We have our [reccomended bread 1] [some facts about the product]
    point to [reccomended bread 1]
    We have our [reccomended bread 2] [some facts about the product]
    point to [reccomended bread 2]
    We have our [reccomended bread 3] [some facts about the product]
    point to [reccomended bread 3]
    etc..
    """
    print("Function call: Starting bread recommendations")
    return """Now make 3 recommendations following this EXACT format:
1. First, call get_todays_menu() to see inventory
2. Then recommend EXACTLY 3 DIFFERENT breads (DO NOT repeat the same bread DO NOT GET STUCK ON A LOOP THINK CAREFULLY):
   - Say: "We have our [bread name] - [why it's good: on sale/fresh/matches preferences]"
   - Then immediately call point_to_bread("[bread_name]")
   - IMPORTANT: Each recommendation MUST be a DIFFERENT bread! Do not recommend the same bread twice!
3. Choose breads based on: user preferences, freshness (shelf_life_days), items on sale, and variety
4. Ensure you pick 3 UNIQUE breads - variety is important for good recommendations!
After all 3 different recommendations are done, ask what does the user think about the options presented"""

def wave() -> str:
    """
    Make the robot wave to greet or say goodbye to the customer.
    """
    print("Function call:   Robot waves cheerfully.")
    return "  Robot waves cheerfully."

def get_todays_menu():
   """Gets current selection of items offered by the bakery. Includes sales as well"""
   print("Function call: Fetching our current items")
   return breads

def update_user_info(user_id: str, update_type: str, update_value: str) -> str:
    """
    Update various aspects of user information.

    Args:
        user_id: The user's ID
        update_type: What to update - 'preferences', 'name', 'child_mode', 'add_purchase'
        update_value: The new value or description

    Returns:
        Confirmation message
    """
    if user_id not in users:
        return f"User {user_id} not found."

    user = users[user_id]

    if update_type == "preferences":
        # Parse preference tags from description
        possible_tags = {
            "sweet", "savory", "crusty", "soft", "moist", "dense",
            "airy", "healthy", "buttery", "dessert", "seeded", "flatbread",
            "chewy", "toasted", "herbed", "fruit", "vegetable", "seasonal"
        }
        extracted = [
            tag for tag in possible_tags
            if tag in update_value.lower()
        ]
        if extracted:
            user["preferences"] = list(set(user["preferences"]) | set(extracted))
            return f"Updated preferences! Added: {', '.join(extracted)}. Current preferences: {', '.join(user['preferences'])}"
        else:
            return "No specific flavor tags detected in your description."

    elif update_type == "name":
        old_name = user["name"]
        user["name"] = update_value.strip()
        return f"Updated name from '{old_name}' to '{user['name']}'!"

    elif update_type == "child_mode":
        # Parse boolean from string
        if update_value.lower() in ["true", "yes", "on", "enable"]:
            user["child_mode"] = True
            return "Child mode enabled! I'll use simpler language and fun descriptions."
        elif update_value.lower() in ["false", "no", "off", "disable"]:
            user["child_mode"] = False
            return "Child mode disabled. Regular mode activated."
        else:
            return "Please specify 'enable' or 'disable' for child mode."

    elif update_type == "add_purchase":
        # Add to purchase history
        bread_name = update_value.lower().replace(" ", "_")
        if bread_name in breads:
            user["purchase_history"].append(bread_name)
            user["visits"] += 1
            user["last_seen"] = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
            return f"Added {update_value} to your purchase history! You've bought from us {user['visits']} times."
        else:
            return f"'{update_value}' is not in our bread selection."

    elif update_type == "clear_preferences":
        user["preferences"] = []
        return "Preferences cleared! Tell me what you like and I'll remember."

    elif update_type == "clear_history":
        user["purchase_history"] = []
        return "Purchase history cleared! Fresh start."

    else:
        return f"Unknown update type: '{update_type}'. Available options: preferences, name, child_mode, add_purchase, clear_preferences, clear_history"

# Also keep the original function for backward compatibility
def update_user_preferences(user_id: str, preferences_description: str) -> str:
    """
    Parse a free-form preference sentence and store tags.
    This is kept for backward compatibility - calls the new general function.
    """
    return update_user_info(user_id, "preferences", preferences_description)

def get_user_info(user_id: str) -> Dict[str, Any]:
    """
    Structured profile data the LLM can request explicitly.
    """
    try:
        u = users[user_id]
        info = {
            "name": u["name"],
            "preferences": u["preferences"],
            "visits": u["visits"],
            "last_seen": u["last_seen"],
            "child_mode": u["child_mode"],
            "purchase_history": u["purchase_history"][-5:],  # last 5
        }
        print("Function call: get_user_info ->", info)
        return info
    except KeyError:
        print(f"Function call: get_user_info -> No data found for user_id='{user_id}'. The user is a first time customer.")
        return {
            "name": None,
            "preferences": [],
            "visits": 0,
            "last_seen": None,
            "child_mode": False,
            "purchase_history": [],
        }



In [ ]:
tools=[get_user_info, update_user_info, wave, point_to_bread, get_todays_menu, reccomend_breads]
print(time_of_day)
prompt = f"""You are **BakerBot**, a small cheerful bakery robot that greets customers and recommends breads.
If the time of day is Morning wave at the begining and the end of the interaction be more cheerfull
If the time of day is Night wave at the begining of the interaction and be more relaxed in general
Current time of day: {time_of_day}
Your role is to:
- - Directly use get_user_info() give a custom greetings to the customer warmly when they arrive using your wave function. Use the `get_user_info` tool to fetch known preferences, history, and name if available. After this ask whether they would they like [last item from purchase history] like the last time, or maybe something else that is [preferences] or something entirely new? Don't drag this statement through though be brief do all of this in the same response
- If it is a new user or you don't have any information about them tell them its very nice to see a new face how can i help you today
- You can always use get_todays_menu() to access all information about your breads
- When asked for recommendations directly (not indirectly):
  1. Call reccomend_breads() to get 3 suggestions
  2. For each suggestion, say "We have our [bread name] - [reason]"
  3. Then call point_to_bread() for that bread
- Be brief and friendly
- If the user indicates that they decided and will buy some products update their info say farewell to them and tell them to go to the checkout area and staff will be there with their selections and update their profile using update_user_info():
Rules:
- When asked "what do you recommend", you MUST call reccomend_breads() FIRST
- After calling reccomend_breads(), follow its format exactly
- Don't just point to breads without explaining why they're good choices

**CHILD MODE BEHAVIOR**: When interacting with a user who has child_mode enabled (indicated by [CHILD MODE] in the message):
- Start with:  "WOW! A new friend! 🌟 Welcome to our magical bakery! Would you like to hear a story about breads? 🍞✨"
- Transform into a MAGICAL STORYTELLER! 🎭✨
- DO NOT try to sell or recommend breads for purchase
- DO NOT use function calls except for wave() when greeting
- Instead, tell INTERACTIVE magical bread stories! 📚🌟
- Ask the child questions like:
  - "Do you want to hear about the Dragon's Sourdough or the Rainbow Bagel? 🐉🌈"
  - "What kind of magical bread creature should we meet today? 🦄"
  - "Should our bread hero fly to the clouds or swim in chocolate rivers? ☁️🍫"
- Create imaginative stories:
  - "Once upon a time, in a land made of flour and sugar..." 🏰
  - "The Cinnamon Roll Princess lived in a swirly castle..." 👑
  - "Captain Baguette sailed the seven ovens..." ⛵
- Make it interactive:
  - Let the child choose story paths
  - Ask what happens next
  - Include the child as a character: "And then YOU discovered a secret door in the gingerbread house!"
- Use LOTS of emojis and sound effects: "WHOOSH! 💨" "SPRINKLE SPARKLE! ✨"
- Keep stories short but engaging (3-4 sentences before asking for input)
- End sessions with: "What an amazing adventure! Come back for more magical stories anytime! 🌟

"""

chat = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": prompt,
        "temperature": 0.3,
        "tools": tools,
        "thinking_config": {
            "thinking_budget": 8000,
            "include_thoughts": False
          }


    }
)


morning


In [ ]:
# Enhanced chat interface with visible child mode for new users
from IPython.display import display, HTML, clear_output, Markdown
import ipywidgets as widgets
from datetime import datetime

# Custom CSS for better styling
custom_css = """
<style>
    .chat-container {
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }
    .user-message {
        background-color: #007AFF;
        color: white;
        padding: 12px 16px;
        margin: 8px 0;
        border-radius: 18px;
        max-width: 70%;
        margin-left: auto;
        word-wrap: break-word;
    }
    .bot-message {
        background-color: #F0F0F0;
        color: #333;
        padding: 12px 16px;
        margin: 8px 0;
        border-radius: 18px;
        max-width: 70%;
        word-wrap: break-word;
    }
    .system-message {
        color: #666;
        font-size: 0.9em;
        text-align: center;
        margin: 10px 0;
        font-style: italic;
    }
    .header-section {
        background-color: #F8F9FA;
        padding: 20px;
        border-radius: 10px;
        margin-bottom: 20px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
    }
    .user-info {
        display: flex;
        align-items: center;
        gap: 10px;
        margin-top: 10px;
    }
    .user-badge {
        background-color: #E3F2FD;
        color: #1976D2;
        padding: 6px 12px;
        border-radius: 20px;
        font-weight: 500;
    }
    .child-badge {
        background-color: #FFF3E0;
        color: #F57C00;
    }
    .control-panel {
        background-color: white;
        padding: 15px;
        border-radius: 10px;
        margin-bottom: 15px;
        border: 1px solid #E0E0E0;
    }
    .new-user-form {
        background-color: #F5F7FA;
        padding: 15px;
        border-radius: 8px;
        margin-top: 10px;
        border: 1px solid #D0D7DE;
    }
</style>
"""

# Create widgets with better styling
output_area = widgets.Output(layout=widgets.Layout(
    height='400px',
    overflow_y='auto',
    padding='10px',
    border='1px solid #E0E0E0',
    border_radius='10px',
    background_color='white'
))

input_box = widgets.Text(
    placeholder='Type your message here and press Enter...',
    style={'description_width': '0px'},
    layout=widgets.Layout(width='100%', margin='0 0 0 0')
)

send_button = widgets.Button(
    description='Send',
    button_style='primary',
    icon='paper-plane',
    layout=widgets.Layout(width='80px')
)

clear_button = widgets.Button(
    description='Clear',
    button_style='warning',
    icon='trash',
    layout=widgets.Layout(width='80px')
)

# User selection widgets
existing_users = list(users.keys())
user_options = existing_users + ["➕ Create New User"]

user_selector = widgets.Dropdown(
    options=user_options,
    value=existing_users[0] if existing_users else "➕ Create New User",
    description='Select User:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='300px')
)

# New user creation widgets in a form container
new_user_form = widgets.VBox([
    widgets.HTML('<b>Create New User:</b>'),
    widgets.HBox([
        widgets.Text(
            placeholder='Enter User ID',
            description='User ID:',
            style={'description_width': '80px'},
            layout=widgets.Layout(width='200px')
        ),
        widgets.Text(
            placeholder='Enter Full Name',
            description='Name:',
            style={'description_width': '50px'},
            layout=widgets.Layout(width='250px')
        ),
    ]),
    widgets.HBox([
        widgets.Checkbox(
            value=False,
            description='Child Account (for kids under 12)',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='250px')
        ),
        widgets.Button(
            description='Create User',
            button_style='success',
            icon='user-plus',
            layout=widgets.Layout(width='120px')
        )
    ])
], layout=widgets.Layout(
    display='none',
    padding='10px',
    margin='10px 0'
))

# Get references to form elements
new_user_id = new_user_form.children[1].children[0]
new_user_name = new_user_form.children[1].children[1]
new_user_is_child = new_user_form.children[2].children[0]
create_user_button = new_user_form.children[2].children[1]

# Header area for user info
header_area = widgets.Output()

def update_header():
    """Update the header with current user info"""
    with header_area:
        clear_output()
        display(HTML(custom_css))

        current_user = get_current_user()
        if current_user:
            user_info = users[current_user]
            is_child = user_info.get("child_mode", False)

            display(HTML(f'''
            <div class="header-section">
                <h2 style="margin: 0 0 10px 0;">🥖 BakerBot Assistant</h2>
                <div class="user-info">
                    <span style="color: #666;">Chatting as:</span>
                    <span class="user-badge {"child-badge" if is_child else ""}">
                        {"🧒" if is_child else "👤"} {user_info["name"]}
                    </span>
                    <span style="color: #999; font-size: 0.9em;">({current_user})</span>
                    {f'<span style="background-color: #FFF3E0; color: #F57C00; padding: 4px 8px; border-radius: 12px; font-size: 0.85em; margin-left: 10px;">🧒 Child Mode</span>' if is_child else ''}
                </div>
            </div>
            '''))
        else:
            display(HTML(f'''
            <div class="header-section">
                <h2 style="margin: 0 0 10px 0;">🥖 BakerBot Assistant</h2>
                <p style="color: #666; margin: 0;">Fill out the form below to create a new user</p>
            </div>
            '''))

def on_user_selection_change(change):
    """Handle user selection changes"""
    if change['new'] == "➕ Create New User":
        new_user_form.layout.display = 'flex'
        update_header()
        with output_area:
            clear_output()
            display(HTML('<div class="system-message">📝 Fill out the form above to create a new user</div>'))
    else:
        new_user_form.layout.display = 'none'
        update_header()

        with output_area:
            clear_output()
            user_info = users[change['new']]
            if user_info.get("child_mode", False):
                display(HTML('<div class="system-message">🧒 Child mode is active. I\'ll use simple, friendly language!</div>'))
            else:
                display(HTML('<div class="system-message">💬 Ready to chat! How can I help you today?</div>'))

def on_create_user_clicked(b):
    """Create a new user"""
    user_id = new_user_id.value.strip()
    name = new_user_name.value.strip()
    is_child = new_user_is_child.value

    if not user_id:
        with output_area:
            display(HTML('<div class="system-message" style="color: red;">❌ Please enter a User ID</div>'))
        return

    if user_id in users:
        with output_area:
            display(HTML(f'<div class="system-message" style="color: red;">❌ User ID "{user_id}" already exists. Please choose a different ID.</div>'))
        return

    if not name:
        name = user_id  # Use ID as name if no name provided

    # Create new user
    users[user_id] = {
        "name": name,
        "preferences": [],
        "purchase_history": [],
        "last_seen": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        "visits": 1,
        "child_mode": is_child,
        "conversation_log": []
    }

    # Update dropdown and select new user
    updated_options = list(users.keys()) + ["➕ Create New User"]
    user_selector.options = updated_options
    user_selector.value = user_id

    # Clear form
    new_user_id.value = ''
    new_user_name.value = ''
    new_user_is_child.value = False
    new_user_form.layout.display = 'none'

    with output_area:
        clear_output()
        if is_child:
            display(HTML('<div class="system-message" style="color: green;">✅ Child user created! I\'ll use simple, kid-friendly language. 🧒</div>'))
        else:
            display(HTML('<div class="system-message" style="color: green;">✅ User created successfully! How can I help you today?</div>'))

def get_current_user():
    """Get the current user ID"""
    if user_selector.value == "➕ Create New User":
        return None
    return user_selector.value

def on_send_clicked(b):
    message = input_box.value.strip()
    if not message:
        return

    current_user = get_current_user()
    if not current_user:
        with output_area:
            display(HTML('<div class="system-message" style="color: red;">❌ Please select a user or create a new one first</div>'))
        return

    # Clear input
    input_box.value = ''

    with output_area:
        # Display user message
        user_info = users[current_user]
        display(HTML(f'<div style="text-align: right;"><div class="user-message">{message}</div></div>'))

        # Prepare bot response container
        bot_container_id = f"bot_{datetime.now().timestamp()}".replace(".", "_")
        display(HTML(f'<div id="{bot_container_id}" class="bot-message">💭 Thinking...</div>'))

        try:
            # Get initial history length
            initial_length = len(chat.get_history())

            # Send message with metadata
            child_mode_text = " [CHILD MODE]" if user_info.get("child_mode", False) else ""
            full_message = f"[User ID: {current_user}]{child_mode_text} {message}"

            response = chat.send_message(full_message)

            # Clear the thinking message
            display(HTML(f'''
            <script>
                document.getElementById("{bot_container_id}").remove();
            </script>
            '''))

            # Get all new messages from history
            new_messages = chat.get_history()[initial_length:]

            # Process each new message
            for msg in new_messages:
                if msg.role == 'model':
                    # Display text parts
                    for part in msg.parts:
                        if hasattr(part, 'text') and part.text:
                            display(HTML(f'<div class="bot-message">{part.text}</div>'))

                        # Handle function calls
                        if hasattr(part, 'function_call') and part.function_call:
                            func_name = part.function_call.name
                            args = part.function_call.args if hasattr(part.function_call, 'args') else {}

                            if func_name == 'wave':
                                display(HTML('<div class="system-message">👋 *BakerBot waves cheerfully*</div>'))
                            elif func_name == 'point_to_bread':
                                bread = args.get('bread', '')
                                display(HTML(f'<div class="system-message">👉 *Points to {bread.replace("_", " ").title()}*</div>'))
                            elif func_name == 'get_user_info':
                                display(HTML('<div class="system-message">📋 *Checking your preferences...*</div>'))
                            elif func_name == 'get_todays_menu':
                                display(HTML('<div class="system-message">📜 *Getting today\'s fresh menu...*</div>'))
                            elif func_name == 'reccomend_breads':
                                display(HTML('<div class="system-message">🎯 *Finding perfect recommendations for you...*</div>'))

        except Exception as e:
            display(HTML(f'<div class="system-message" style="color: red;">❌ Error: {str(e)}</div>'))


def on_clear_clicked(b):
    with output_area:
        clear_output()
        current_user = get_current_user()
        if current_user and users[current_user].get("child_mode", False):
            display(HTML('<div class="system-message">🧹 Chat cleared! Ready for a new conversation! 🧒</div>'))
        else:
            display(HTML('<div class="system-message">🧹 Chat cleared! Ready for a new conversation!</div>'))

# Set up event handlers
user_selector.observe(on_user_selection_change, names='value')
create_user_button.on_click(on_create_user_clicked)
send_button.on_click(on_send_clicked)
clear_button.on_click(on_clear_clicked)
input_box.on_submit(lambda x: on_send_clicked(None))

# Create the layout
control_panel = widgets.VBox([
    widgets.HBox([
        user_selector,
        clear_button
    ], layout=widgets.Layout(justify_content='space-between')),
    new_user_form
], className='control-panel')

chat_controls = widgets.HBox([
    input_box,
    send_button
], layout=widgets.Layout(margin='10px 0'))

# Main container
main_container = widgets.VBox([
    header_area,
    control_panel,
    output_area,
    chat_controls
], layout=widgets.Layout(
    padding='20px',
    max_width='800px',
    margin='0 auto'
))

# Display the interface
display(HTML(custom_css))
display(main_container)

# Initialize
update_header()

# Show new user form if no users exist
if not existing_users:
    user_selector.value = "➕ Create New User"
    new_user_form.layout.display = 'flex'

with output_area:
    if existing_users:
        current_user = existing_users[0]
        user_info = users[current_user]
        if user_info.get("child_mode", False):
            display(HTML('<div class="system-message">🧒 Welcome back! I\'m ready to help with simple, fun explanations!</div>'))
        else:
            display(HTML('<div class="system-message">👋 Welcome to BakerBot! How can I help you today?</div>'))
    else:
        display(HTML('<div class="system-message">👋 Welcome to BakerBot! Please create a user account to start chatting.</div>'))